# Phase 1 — Reddit Engagement Analysis

**Instructor:** Prof. Alexis Yanez  
**Due:** Tuesday, March 3rd, 2026

### Student Information
- **Name:** Affan Thameem  
- **Student ID:** 40282375
- **Section / Lab:** COMP 333 - GJ-X 
---
- **Name:** Mohammad Salah Sartawi
- **Student ID:** 40246396
- **Section / Lab:** COMP 333 - GJ-X
---
- **Name:** Chimdindu Okelekwe
- **Student ID:** 40153875
- **Section / Lab:**  COMP 333 - GI-X 

#### Objective & Research Questions

**RQ1 (Supervised):** Can we predict **high engagement** (score ≥ 80th percentile) from content and structural features?
- **Features:** `title_length`, `has_selftext`, `subreddit`, `post_type`, `hour_of_day`, `day_of_week`, `is_weekend`, `domain`

**RQ2 (Unsupervised):** Can we discover natural clusters of Reddit submissions based on engagement patterns, subreddit, and content structure  **without using the score label**?
- **Features:** `title_length`, `has_selftext`, `subreddit`, `post_type`, `hour_of_day`, `day_of_week`, `is_weekend`, `domain`


## 1. Prerequisites

1. Import all packages

#### Datasets
- Reddit comments/submissions 2026-01: https://academictorrents.com/details/8412b89151101d88c915334c45d9c223169a1a60
- Access dataset: https://transmissionbt.com/

---

#### Dataset Setup

1. Download the dataset zip file from the link below:

   -> [Download Dataset (Google Drive)](https://drive.google.com/file/d/1W6mqYfel_PRtVGaikCB-HepsuLJ4buWT/view)

2. Unzip the downloaded file to find `reddit_subset.jsonl` at the same level as the notebook:

```
project/
└── reddit_subset.jsonl
└── Phase_1.ipynb
```

---

#### Running the Notebook

Open `Phase_1.ipynb` in Jupyter and run all cells top to bottom.

> **Note:** The data extraction cell (Section 1.1) is already commented out, the dataset is pre-extracted. Do not uncomment or re-run it.

---

#### Notebook Structure

| Section | Description |
|---------|-------------|
| 1. Data Retrieval | Documents how the raw dataset was sourced and extracted |
| 2. Wrangling & Cleaning | Cleaning pipeline, feature engineering, and data quality checks |
| 3. EDA | Exploratory analysis of features and engagement patterns |
| 4. Baseline Model | Logistic regression baseline; train/val/test split; evaluation metrics |
| 5. Report | Summary of work and division of labour |

In [8]:
import zstandard as zstd
import json
import io
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import re
import gdown
import zipfile
import os

pd.set_option('display.float_format', '{:.2f}'.format)

### 1.1 Data Retrieval (Document Sources; Programmatic Retrieval; Challenges; Raw Storage)

#### **Document sources**

*   **Primary source:** Reddit submissions (January 2026) obtained from [Academic Torrents](https://academictorrents.com/details/8412b89151101d88c915334c45d9c223169a1a60) (Pushshift/Reddit data dump).
    
*   **Access method:** BitTorrent client (e.g., [Transmission](https://transmissionbt.com/)) to download the compressed `RS_2026-01.zst` file (~19.6 GB).
    

#### **Programmatic retrieval**

*   We use **file-based streaming extraction** (no API or SQL queries). 
*   The Python `zstandard` library decompresses the `.zst` archive and streams the data line-by-line.  
*   Each line represents one Reddit submission in JSON format.
*   During streaming:
    
    *   JSON lines are parsed using json.loads().
        
    *   NSFW posts (over\_18 = True) are excluded.
        
    *   Only a predefined subset of relevant fields is retained.
        
    *   Extraction stops once a ~2 GB subset is written to disk.
        

This streaming approach assures memory efficiency and avoids loading the full 19.6 GB archive into RAM.

#### **Handling challenges**

*   **Rate limits:** Not applicable - data is downloaded via torrent and processed locally.
    
*   **Authentication:** Not required - dataset is publicly available.
    
*   **Data quality:**
    
    *   NSFW content is filtered during extraction.
        
    *   Only fields required for research questions are retained to reduce noise and storage overhead.
        

#### **Raw data storage**

*   Output file: reddit\_subset.jsonl (~2 GB).
    
*   Format: JSON Lines (one Reddit submission per line).
    
*   The stored dataset does **not** preserve all original 152 fields; instead, it retains only the subset necessary for analysis:
    
    *   id
        
    *   subreddit
        
    *   created\_utc
        
    *   title
        
    *   selftext
        
    *   is\_self
        
    *   is\_video
        
    *   domain
        
    *   score
        

This storage strategy:

*   Ensures reproducibility while reducing unnecessary metadata.
    
*   Prevents data leakage by excluding post-engagement fields (e.g., comments, awards).
    
*   Preserves all variables required to derive structural and contextual features for both supervised (RQ1) and unsupervised (RQ2) analyses.
    
*   Maintains a manageable dataset size for local computation while remaining statistically representative.


**NO NEED TO RUN AGAIN**

In [ ]:
# INPUT_FILE_PATH = "../Dataset/RS_2026-01.zst"
# OUTPUT_FILE_PATH = "reddit_subset.jsonl"
# TARGET_SIZE_BYTES = 2 * 1024**3  # 2GB

# FIELDS_TO_KEEP = [
#     "id",           # Unique post identifier (used for tracking/deduplication, not as a feature)
#     "subreddit",    # Captures community context and engagement norms
#     "created_utc",  # Used to derive time-based features (hour, day, weekend)
#     "title",        # Used to compute title length and optional text features
#     "selftext",     # Used to detect content presence and optional text features
#     "is_self",      # Indicates text post vs link post (post type feature)
#     "is_video",     # Indicates video content (post type feature)
#     "score",        # Used only to create high-engagement label (not as input feature)
#     "domain",       # Identifies external source platform for refining post type
# ]

# bytes_written = 0
# rows_written = 0

# with open(INPUT_FILE_PATH, "rb") as compressed_file:
#     decompressor = zstd.ZstdDecompressor()

#     with decompressor.stream_reader(compressed_file) as decompressed_stream:
#         text_stream = io.TextIOWrapper(decompressed_stream, encoding="utf-8", errors="replace")

#         with open(OUTPUT_FILE_PATH, "w", encoding="utf-8") as output_file:
#             for raw_line in tqdm(text_stream, desc="Extracting submissions"):
#                 submission = json.loads(raw_line)  # Will raise if malformed (handled in Part 2)

#                 # Filter out NSFW content
#                 if submission.get("over_18"):
#                     continue

#                 filtered_submission = {field: submission.get(field) for field in FIELDS_TO_KEEP}

#                 json_line = json.dumps(filtered_submission, ensure_ascii=False) + "\n"
#                 output_file.write(json_line)

#                 bytes_written += len(json_line.encode("utf-8"))
#                 rows_written += 1

#                 if bytes_written >= TARGET_SIZE_BYTES:
#                     break

# print("Done! Extracted ~2GB subset.")
# print(f"Rows written: {rows_written:,}")
# print(f"Total written: {bytes_written / (1024**3):.2f} GB")

##### Due to the large size of the dataset (~2.15GB), smaller subsets (500MB, 1GB, and 1.5GB) were created for efficient experimentation and demonstration, while preserving the original data format.

In [ ]:
# def create_random_subsets(input_file, output_file, sample_ratio=0.25):
#     """
#     Random sampling for large JSONL files
#     """
#     with open(input_file, "r", encoding="utf-8") as infile, \
#          open(output_file, "w", encoding="utf-8") as outfile:

#         lines_written = 0

#         for line in infile:
#             if random.random() < sample_ratio:
#                 outfile.write(line)
#                 lines_written += 1

#     print(f"Random subset saved: {output_file}")
#     print(f"Lines written: {lines_written:,}")

In [ ]:
# create_random_subsets("reddit_subset.jsonl", "reddit_500MB.jsonl", 0.25)
# create_random_subsets("reddit_subset.jsonl", "reddit_1GB.jsonl", 0.5)
# create_random_subsets("reddit_subset.jsonl", "reddit_1_5GB.jsonl", 0.7)

#### Select data set you with to run.

[dataset1 - 2.15 GB](https://drive.google.com/file/d/1W6mqYfel_PRtVGaikCB-HepsuLJ4buWT/view?usp=sharing): ~3.85M Posts

[dataset2 - 1.5 GB](https://drive.google.com/file/d/1uK1CG1nWNu00fkIQO5zrJbC02sITj7U7/view?usp=sharing): ~2.69M Posts

[dataset3 - 1 GB](https://drive.google.com/file/d/19afx8E9rm5nGQ3isregVp6Y_PwGq6tEx/view?usp=sharing): ~1.92M Posts

[dataset4 - 0.5 GB](https://drive.google.com/file/d/13TXBzK0r22c67FuWl7QQCCdvccXARQ1L/view?usp=sharing): ~0.96M Posts

In [14]:
def download_and_prepare_dataset(mode):
    datasets = {
        "full": {
            "name": "2.15GB",
            "file_id": "1W6mqYfel_PRtVGaikCB-HepsuLJ4buWT",
            "filename": "reddit_subset.zip",
            "is_zip": True
        },
        "large": {
            "name": "1.5GB",
            "file_id": "1uK1CG1nWNu00fkIQO5zrJbC02sITj7U7",
            "filename": "reddit_1_5GB.jsonl",
            "is_zip": False
        },
        "medium": {
            "name": "1GB",
            "file_id": "19afx8E9rm5nGQ3isregVp6Y_PwGq6tEx",
            "filename": "reddit_1GB.jsonl",
            "is_zip": False
        },
        "demo": {
            "name": "500MB",
            "file_id": "13TXBzK0r22c67FuWl7QQCCdvccXARQ1L",
            "filename": "reddit_500MB.jsonl",
            "is_zip": False
        }
    }

    if mode not in datasets:
        raise ValueError("Invalid mode")

    dataset = datasets[mode]
    filename = dataset["filename"]

    # 2. Extract if ZIP
    if dataset["is_zip"]:
        extracted_jsonl = None
        if os.path.exists(filename):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                for file in zip_ref.namelist():
                    if file.endswith(".jsonl") and os.path.exists(file):
                        extracted_jsonl = file
                        break

        if extracted_jsonl:
            print(f" Already extracted. Using: {extracted_jsonl}")
            return extracted_jsonl

        # Download if zip not present
        if not os.path.exists(filename):
            print(f" Downloading {dataset['name']} dataset...")
            url = f"https://drive.google.com/uc?id={dataset['file_id']}"
            gdown.download(url, filename, quiet=False)

        print(" Extracting ZIP...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall()
            for file in zip_ref.namelist():
                if file.endswith(".jsonl"):
                    print(f" Using: {file}")
                    return file

    else:
        # For plain .jsonl files, skip download if file already exists
        if os.path.exists(filename):
            print(f" File already exists. Using: {filename}")
            return filename

        print(f" Downloading {dataset['name']} dataset...")
        url = f"https://drive.google.com/uc?id={dataset['file_id']}"
        gdown.download(url, filename, quiet=False)

    print(f" Using: {filename}")
    return filename

In [17]:
# mode="full"     2.15GB 
# mode="large"    1.5GB
# mode="medium"   1GB
# mode="demo"     500MB

INPUT_FILE = download_and_prepare_dataset("large")

 File already exists. Using: reddit_1_5GB.jsonl


---

## 2. Wrangling & Cleaning

This section performs an **initial audit** (missing values, duplicates, outliers) and applies a **reproducible cleaning pipeline**. We use `quantDDA()` and `vizDDA()` from Lab 2 for quantitative and visual data quality checks.

### 2.1 Initial Audit (Missing, Duplicates, Outliers)

In [ ]:
def audit_dataframe(df):
    """Initial audit: missing values, duplicates, basic outlier context."""
    return {
        "shape": df.shape,
        "missing": df.isnull().sum().sort_values(ascending=False),
        "missing_pct": (df.isnull().mean() * 100).round(2).sort_values(ascending=False),
        "duplicate_ids": df["id"].duplicated().sum() if "id" in df.columns else 0,
        "dtypes": df.dtypes
    }

df = pd.read_json(INPUT_FILE, lines=True)
audit = audit_dataframe(df)
print("Shape:", audit["shape"])

print("\n=== COLUMN DTYPES ===")
print(audit["dtypes"])

print("\n=== TOP MISSING VALUES (count) ===")
print(audit["missing"].head(20))

print("\n=== TOP MISSING VALUES (%) ===")
print(audit["missing_pct"].head(20))

print("\n=== DUPLICATES ===")
print("Duplicate IDs:", audit["duplicate_ids"])

# Score distribution checks (target variable source)
print("\n=== SCORE DESCRIBE ===")
print(df["score"].describe())

print("\n=== SCORE QUANTILES ===")
print(df["score"].quantile([0.5, 0.75, 0.8, 0.9, 0.95]))

print("\n=== FIRST 20 ROWS ===")
display(df.head(20))

---

### 2.2 Reproducible Cleaning Pipeline (RQ1 & RQ2)

For consistency and reproducibility across both research questions, we implemented preprocessing function, `clean_reddit_data()`, which normalizes the dataset and engineers structural features required for analysis.

The raw dataset contains the following retained fields:`id`, `subreddit`, `created\_utc`, `title`, `selftext`, `is_self`, `is_video`, `domain`, and `score`.

From these raw fields, additional columns are created:

*   `high_engagement` -> binary target variable defined as score ≥ 80th percentile
    
*   `title_length` -> character length of the submission title
    
*   `has_selftext` -> binary indicator of whether body content is present
    
*   `post_type` -> categorical feature derived from is\_self and is\_video (self, video, or link)
    
*   `hour_of_day` -> UTC hour extracted from created\_utc
    
*   `day_of_week` -> weekday index (0–6)
    
*   `is_weekend` -> binary weekend indicator
    

#### Target Definition (RQ1 - Supervised Learning)

Reddit engagement follows a heavy right-skewed distribution. To reduce skew effects and produce a stable classification task, high engagement is defined as submissions with a score at or above the 80th percentile of the score distribution. This percentile-based thresholding produces the binary variable `high_engagement`.

#### RQ1 (Supervised Modeling)

The engineered structural features (`title_length`, `has_selftext`, `subreddit`, `post_type`, `hour_of_day`, `day_of_week`, `is_weekend`, and `domain`) are used to predict the binary outcome `high_engagement`.

#### RQ2 (Unsupervised Clustering)

The same structural and contextual features are used for clustering analysis. The engagement variable (`score`) and derived label (`high_engagement`) are excluded from clustering inputs to prevent information leakage and avoid biased or inflated evaluation outcomes. After forming clusters without using score, we examine how engagement varies across clusters to understand what each cluster represents.

#### Reproducibility Guarantees

The cleaning pipeline ensures reproducibility by:

*   Applying deterministic feature transformations
    
*   Explicitly defining derived columns
    
*   Handling missing values consistently
    
*   Removing duplicate submission IDs
    
*   Using identical preprocessing logic for both RQ1 and RQ2

In [ ]:
# Reproducible pipeline: features aligned with RQ1 (supervised) and RQ2 (unsupervised)
def clean_reddit_data(raw_path_or_df, engagement_quantile: float = 0.80) -> pd.DataFrame:
    """Clean Reddit dataset + create label + engineer features (leak-free safe stage)."""

    raw_df = (pd.read_json(raw_path_or_df, lines=True) if isinstance(raw_path_or_df, str) else raw_path_or_df.copy())

    df = raw_df.copy()
    
    # 1. Basic normalization
    df["title"] = df["title"].fillna("").astype(str).str.strip()
    df["selftext"] = df["selftext"].fillna("").astype(str).str.strip()
    df["domain"] = df["domain"].fillna("")

    # Remove empty titles 
    df = df[df["title"] != ""]

    # 2. Fix Reddit-specific missing values
    def is_invalid_text(text):
        text = str(text).lower().strip()
        text = re.sub(r'[\[\](){}]', '', text)  # remove brackets
        text = re.sub(r'\s+', ' ', text)        # normalize spaces

        return text == "" or "removed" in text or "deleted" in text
    
    df = df[~df["title"].apply(is_invalid_text)]
    df["has_selftext"] = ~df["selftext"].apply(is_invalid_text)
    df["has_selftext"] = df["has_selftext"].astype(int)

    # 3. Target label (RQ1)
    threshold = df["score"].quantile(engagement_quantile)
    df["high_engagement"] = (df["score"] >= threshold).astype(int)

    # 4. Feature engineering
    df["title_length"] = df["title"].str.len()

    # Updated: content type (replaces post_type)
    df["content_type"] = np.select(
        [
            df["is_self"].astype(bool),
            df["is_video"].astype(bool),
            df["domain"].str.contains("i.redd.it", na=False),
            df["domain"].str.contains("v.redd.it", na=False),
        ],
        [
            "text",
            "video",
            "image",
            "video",  # reddit video domain
        ],
        default="link"
    )

    # Time features
    created_dt = pd.to_datetime(df["created_utc"], unit="s", utc=True, errors="coerce")
    df["hour_of_day"] = created_dt.dt.hour
    df["day_of_week"] = created_dt.dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    # 5. Handle outliers
    def cap_outliers(series, lower_q=0.01, upper_q=0.99):
        lower = series.quantile(lower_q)
        upper = series.quantile(upper_q)
        return series.clip(lower, upper)

    df["score"] = cap_outliers(df["score"])
    df["title_length"] = cap_outliers(df["title_length"])


    # 6. Remove duplicates
    df = df.drop_duplicates(subset=["id"], keep="first")


    # 7. Final cleanup
    df = df.dropna(subset=["hour_of_day", "day_of_week"])

    return df

def get_rq2_features(df_clean: pd.DataFrame) -> pd.DataFrame:
    """RQ2 clustering matrix: excludes score + label."""
    rq2_cols = [
        "subreddit",
        "title_length",
        "has_selftext",
        "content_type",  # updated
        "hour_of_day",
        "day_of_week",
        "is_weekend",
        "domain"
    ]
    return df_clean[rq2_cols].copy()


# Run cleaning pipeline on the full dataset
df_clean = clean_reddit_data(INPUT_FILE, engagement_quantile=0.80)
threshold = df_clean["score"].quantile(0.80)

print("Cleaned shape:", df_clean.shape)
print("High engagement threshold (80th %):", threshold)
print("Target distribution:", df_clean["high_engagement"].value_counts(normalize=True).to_dict())
print("Columns:", df_clean.columns.tolist())

df_rq2 = get_rq2_features(df_clean)
print("RQ2 feature shape:", df_rq2.shape)


### 2.3 quantDDA() and vizDDA() (from Lab 2)

We use the `quantDDA()` and `vizDDA()` functions from Lab 2 for quantitative and visual data quality checks. `quantDDA` reports missing values, outliers (IQR), skewness, kurtosis; `vizDDA` produces histograms, scatter plots, boxplots, and heatmaps.

In [ ]:
def quantDDA(df):
    summary_list = []  # List to store summary dictionaries

    for col in df.columns:  # Loop through each column in the DataFrame
        series = df[col]  
        
        # Basic counts (for all features)
        n_obs = len(series)                     # Total number of observations (rows)
        n_entries = series.count()              # Number of non-missing values (entries)
        n_unique = series.nunique(dropna=True)  # Number of unique non-missing values
        n_missing = series.isna().sum()         # Number of missing (NaN) values

        # Mode(s): most frequent value(s)
        if n_unique == n_entries:  
            modes = "No mode (all values unique)"  # No mode if every value appears once
        else:
            modes = series.dropna().mode().tolist()  # Most frequent value(s), ignoring NaN
            
        # Dictionary with universal statistics
        summary = {
            "Feature": col,                     # Feature (column) name
            "Observations": int(n_obs),         # Total observations in the dataset
            "Entries": int(n_entries),          # Non-missing values in this feature
            "Unique": int(n_unique),            # Unique values count
            "Missing": int(n_missing),          # Missing values count
            "Mode(s)": modes                    # Most common value(s)
        }

        # Compute numeric statistics only for numeric columns
        if pd.api.types.is_numeric_dtype(series) and not pd.api.types.is_bool_dtype(series):   # Check if column contains numbers
            clean = series.dropna()                 # Remove missing values

            if len(clean) > 0:                      # Only compute if column is not empty

                # Quartiles
                q1 = clean.quantile(0.25)           # First quartile (25th percentile)
                q2 = clean.quantile(0.50)           # Median (50th percentile)
                q3 = clean.quantile(0.75)           # Third quartile (75th percentile)

                # IQR Outlier bounds
                iqr = q3 - q1                       # Interquartile range
                lower = q1 - 1.5 * iqr              # Lower outlier boundary
                upper = q3 + 1.5 * iqr              # Upper outlier boundary

                # Extreme value thresholds (top/bottom 1%)
                lower_ext = clean.quantile(0.01)    # Bottom 1% threshold
                upper_ext = clean.quantile(0.99)    # Top 1% threshold

                # Update dictionary with numeric-only stats
                summary.update({
                    "Mean": clean.mean(),           # Average value
                    "Standard Deviation": clean.std(),             # Standard deviation
                    "Minimum": clean.min(),             # Minimum value
                    "Maximum": clean.max(),             # Maximum value
                    "First Quartile": q1,                       # 25th percentile
                    "Median": q2,              # Median (50th percentile)
                    "Third Quartile": q3,                       # 75th percentile
                    "Skewness": clean.skew() if len(clean) > 2 else np.nan,                     #  Measures distribution asymmetry
                    "Kurtosis": clean.kurt() if len(clean) > 3 else np.nan,                     # Measures tail heaviness / peakedness
                    "Outliers": int(((clean < lower) | (clean > upper)).sum()),                 # Values outside IQR boundaries
                    "Extreme (1%)": int(((clean <= lower_ext) | (clean >= upper_ext)).sum())    # Values in top/bottom 1% of distribution
                })

        summary_list.append(summary) 

    return pd.DataFrame(summary_list)


In [ ]:
# Audit RQ1/RQ2 numeric features
quantDDA(df_clean)

In [ ]:
def vizDDA(name, df):
    print(f"\n{'=' * 80}\n{name}\n{'=' * 80}")

    df_plot = df.copy()

    max_unique = 10  # Set to 500 to see all columns. Set to 10 for visual purposes (only low-cardinality categorical). Adjust as needed.

    def safe_nunique(s):
        try:
            return s.nunique(dropna=True)
        except (TypeError, ValueError):
            return float('inf')  # Skip columns with unhashable types

    keep_cols = [
        col for col in df_plot.columns
        if pd.api.types.is_numeric_dtype(df_plot[col]) or safe_nunique(df_plot[col]) <= max_unique
    ]  # Keep numeric always + only low-cardinality categorical

    df_plot = df_plot[keep_cols]  # Filter df for readable plotting

    cols = df_plot.columns.tolist()  # Column names used in grid
    n = len(cols)  # Grid size (n x n)

    numeric_cols = df_plot.select_dtypes(include=[np.number]).columns.tolist()  # Numeric columns
    categorical_cols = [c for c in cols if c not in numeric_cols]  # Categorical columns (low-cardinality)

    print("Columns used in grid:", cols)

    fig, axes = plt.subplots(n, n, figsize=(4 * n, 4 * n))  # Create plot grid
    if n == 1:
        axes = np.array([[axes]]) 

    # Loop through grid
    for i in range(n):
        for j in range(n):
            ax = axes[i, j]  # Current subplot
            row_var = cols[i]  # Y variable
            col_var = cols[j]  # X variable

            # Matrix headers 
            if i == 0:
                ax.set_title(col_var)  # Column headers
            if j == 0:
                ax.set_ylabel(row_var)  # Row headers
            else:
                ax.set_ylabel("")  # Reduce clutter

            if i != n - 1:
                ax.set_xlabel("")  # Hide x-label except bottom row

            # Diagonal: Univariate
            if i == j:
                if row_var in numeric_cols:
                    df_plot[row_var].dropna().hist(ax=ax, bins=30, edgecolor="black")
                    cell_label = f"{row_var} (hist)"  # Cell meaning
                else:
                    df_plot[row_var].value_counts().plot(kind="bar", ax=ax)
                    ax.tick_params(axis="x", rotation=45)
                    cell_label = f"{row_var} (bar)"  # Cell meaning

            # Off-diagonal: Bivariate
            else:
                # Numeric vs Numeric -> Scatter
                if row_var in numeric_cols and col_var in numeric_cols:
                    ax.scatter(df_plot[col_var], df_plot[row_var], alpha=0.5, s=15)
                    cell_label = f"{row_var} vs {col_var} (scatter)"  # Cell meaning

                # Numeric vs Categorical -> Boxplot
                elif row_var in numeric_cols and col_var in categorical_cols:
                    sns.boxplot(x=df_plot[col_var], y=df_plot[row_var], ax=ax)
                    ax.tick_params(axis="x", rotation=45)
                    cell_label = f"{row_var} by {col_var} (box)"  # Cell meaning

                # Categorical vs Numeric -> Boxplot (variables flipped)
                elif row_var in categorical_cols and col_var in numeric_cols:
                    sns.boxplot(x=df_plot[row_var], y=df_plot[col_var], ax=ax)
                    ax.tick_params(axis="x", rotation=45)
                    cell_label = f"{col_var} by {row_var} (box)"  # Cell meaning

                # Categorical vs Categorical -> Heatmap
                else:
                    ct = pd.crosstab(df_plot[row_var], df_plot[col_var])
                    sns.heatmap(
                        ct,
                        annot=True,
                        fmt="d",
                        cbar=False,
                        ax=ax,
                        annot_kws={"color": "white"}  # Heatmap numbers in white
                    )
                    ax.tick_params(axis="x", rotation=45)
                    cell_label = f"{row_var} x {col_var} (count)"  # Cell meaning

            # Labeling
            ax.text(
                0.02, 0.95, cell_label,  # Position inside axes (top-left)
                transform=ax.transAxes,
                fontsize=9,
                va="top",
                ha="left",
                bbox=dict(facecolor="white", alpha=0.7, edgecolor="none") # styling
            )

    plt.tight_layout()
    plt.show()

    # Missing values heatmap
    plt.figure(figsize=(10, 6))
    sns.heatmap(df_plot.isna(), cbar=False)
    plt.title("Missing Values Heatmap")
    plt.xlabel("Columns")
    plt.ylabel("Rows")
    plt.tight_layout()
    plt.show()


In [ ]:
# Viz subset: RQ1/RQ2 
cols_for_viz = [
    "score", "high_engagement",
    "title_length", "has_selftext",
    "hour_of_day", "day_of_week", "is_weekend",
    "post_type"
]
vizDDA("DDA Visual Diagnostics (key features)", df_clean[cols_for_viz].sample(20000, random_state=42))

## 3. Exploratory Data Analysis (EDA)

### 3.1 Purpose

After completing data retrieval and cleaning, we now explore the dataset to better understand its structure before implementing any predictive models.

The objectives of this EDA are to:

- Examine the statistical properties of the engineered features  
- Analyze how structural attributes relate to engagement  
- Assess the balance of the target variable  
- Identify patterns that may support both supervised and unsupervised learning  

All variables analyzed in this section were generated in `clean_reddit_data()` and are aligned with the research questions defined earlier.

---

### 3.2 Summary Statistics

Using `quantDDA()`, we computed descriptive statistics for the main modeling variables:

- `score`
- `high_engagement`
- `title_length`
- `has_selftext`
- `hour_of_day`
- `day_of_week`

The dataset contains **3,848,496 submissions**, and no missing values were found in the selected model features. This confirms that the cleaning pipeline successfully produced a modeling-ready dataset.

### Score Distribution (Context)

From the audit results:

- Median score = 1  
- 75th percentile = 8  
- 80th percentile = 14  
- 90th percentile = 54  
- 95th percentile = 146  
- Maximum score = 169,757  

The distribution of scores is extremely right-skewed. Most posts receive minimal engagement, while a small number receive very high scores. This heavy-tailed behavior is typical of online content platforms and supports the decision to define high engagement using a percentile-based threshold rather than a fixed value.

---

### 3.3 Target Variable Distribution

The binary target `high_engagement` labels posts above the 80th percentile as 1.

As expected:

- Approximately 20% of posts are labeled high engagement  
- Approximately 80% are labeled low engagement  

This introduces moderate class imbalance. While not extreme, it suggests that evaluation metrics such as precision, recall, and F1-score will be more informative than accuracy alone in Phase 2.

---

### 3.4 Univariate Analysis

We use `vizDDA()` to examine the distribution of individual features.

### Title Length

Title length exhibits noticeable variability, with a right-skewed distribution. Most titles are relatively short, but there is a long tail of longer titles.

This variability indicates that title length meaningfully differentiates posts and may capture differences in content detail or intent.

---

### Has Selftext

`has_selftext` is a binary indicator of whether a post includes body text.

This variable separates link-style posts from discussion-style posts. Since format influences how users interact with content, this structural distinction may play an important role in predicting engagement.

---

### Post Type

Post type (`self` vs `link`) reinforces the structural distinction between content formats.

Differences in engagement patterns across formats suggest that user behavior may vary depending on whether a post invites discussion or redirects users to external content.

---

### Hour of Day

Posting activity is not uniformly distributed across all hours. Some hours show significantly higher activity levels.

This suggests that user engagement on Reddit follows daily behavioral cycles. Time-of-day may therefore contribute predictive value.

---

### Day of Week

Although weekly variation appears less pronounced than daily variation, the inclusion of `day_of_week` allows the model to capture potential cyclical engagement patterns.

---

### 3.5 Bivariate Analysis

We now examine how structural features relate to engagement.

### Title Length vs High Engagement

Boxplots indicate that high-engagement posts show slightly greater variability in title length. While there is substantial overlap between classes, longer titles appear somewhat more frequently among high-engagement posts.

This suggests that title length alone is not a strong discriminator, but may contribute predictive signal when combined with other features.

---

### Has Selftext vs Engagement

Engagement rates differ between posts with and without selftext. Posts containing selftext appear to follow different engagement patterns than link-only posts.

This supports the hypothesis that content format influences user interaction.

---

### Post Type vs Engagement

Differences between self and link posts are also reflected in engagement distributions. This reinforces the importance of structural features in modeling engagement.

---

### Subreddit vs Engagement

Grouping the top subreddits reveals noticeable variation in engagement rates across communities.

This suggests that subreddit context captures community-level effects such as audience size, norms, and activity levels. Subreddit is therefore likely to be one of the strongest predictors for RQ1 and a key variable in clustering for RQ2.

---

### 3.6 Correlation Analysis

To assess relationships among numeric predictors, we computed the correlation matrix for:

- `score`
- `title_length`
- `hour_of_day`
- `day_of_week`

The correlations between structural features are generally weak.

This indicates:

- Low multicollinearity among predictors  
- Each feature contributes relatively independent information  
- Engagement behavior is unlikely to be explained by simple linear relationships alone  

This supports the use of classification models that can capture nonlinear interactions in Phase 2.

---

### 3.7 Data Quality Confirmation

The missing values heatmap generated by `vizDDA()` shows no missing values in the selected modeling variables.

This confirms that the cleaning pipeline successfully handled missing data and that no additional preprocessing is required before baseline modeling.

---

### 3.8 Summary of Insights

From this exploratory analysis:

1. Engagement is intentionally imbalanced (~20% high engagement).  
2. Score distribution is highly skewed, validating percentile-based thresholding.  
3. Structural features (title length, selftext presence, post type) show meaningful variability.  
4. Temporal features capture daily behavioral patterns.  
5. Subreddit appears to strongly influence engagement.  
6. Numeric predictors exhibit low correlation, reducing concerns about redundancy.  

Overall, the dataset demonstrates sufficient structure and variability to support both:

- Binary classification (RQ1)  
- Clustering based on structural attributes (RQ2)  

This EDA provides a clear foundation for implementing and evaluating the baseline model in the next section.

---

---
## 4. Baseline Model

We address **RQ1 (Supervised):** predicting `high_engagement` (binary: score ≥ 80th percentile) from content and structural features.

**Dataset size:** 3,848,496 rows after cleaning (confirmed in Section 2).

**Target:** `high_engagement` (1 = score ≥ 80th percentile, 0 = otherwise) — approximately **20.5% positive class**.

**Model choice:** Logistic Regression.  
As a baseline it is interpretable via coefficients, fast to train on large data, and well-suited to binary classification. We use `class_weight='balanced'` to account for the ~4:1 class imbalance.

---

### 4.1 Feature Engineering for Modeling

The full RQ1 feature set (defined in Section 1) is:

| Feature | Type | Encoding |
|---|---|---|
| `title_length` | Numeric | StandardScaler |
| `has_selftext` | Binary numeric | As-is |
| `hour_of_day` | Numeric | StandardScaler |
| `day_of_week` | Numeric | StandardScaler |
| `is_weekend` | Binary numeric | As-is |
| `post_type` | Categorical (3 levels) | One-hot (drop first) |
| `subreddit` | High-cardinality categorical | Top-50 + "Other" → One-hot |
| `domain` | High-cardinality categorical | Top-50 + "Other" → One-hot |

**Handling high-cardinality categoricals:** `subreddit` has 195,490 unique values and `domain` has 183,528. One-hot encoding all levels would produce an intractably wide matrix and overfit to rare communities. We retain the **top 50 most frequent** values for each and bin the rest as `"Other"`, giving a manageable 100 additional binary columns that still capture the vast majority of signal (top-50 subreddits account for the bulk of posting volume).

---

### 4.2 Train / Validation / Test Split

We use a **70 / 15 / 15** split, stratified on the target to preserve the class ratio across all three sets.

**Justification:** With ~3.85M rows the smallest split (15% ≈ 577k rows) is orders of magnitude larger than needed for reliable estimation, so the default proportions are appropriate.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve
)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ── 1. Build full feature matrix ─────────────────────────────────────────────

TOP_N = 50  # Number of top levels to keep for high-cardinality features

NUMERIC_FEATURES   = ["title_length", "has_selftext", "hour_of_day", "day_of_week", "is_weekend"]
CATEGORICAL_LOW    = ["post_type"]                   # 3 levels — encode directly
CATEGORICAL_HIGH   = ["subreddit", "domain"]         # 195k / 183k levels — cap at top-50

TARGET = "high_engagement"

df_bl = df_clean[NUMERIC_FEATURES + CATEGORICAL_LOW + CATEGORICAL_HIGH + [TARGET]].copy()

# Cap high-cardinality columns at top-N, rest → "Other"
for col in CATEGORICAL_HIGH:
    top_vals = df_bl[col].value_counts().nlargest(TOP_N).index
    df_bl[col] = df_bl[col].where(df_bl[col].isin(top_vals), other="Other")

# One-hot encode all categorical features (drop first to avoid dummy trap)
df_bl = pd.get_dummies(df_bl, columns=CATEGORICAL_LOW + CATEGORICAL_HIGH, drop_first=True)

X = df_bl.drop(columns=[TARGET])
y = df_bl[TARGET]

print(f"Feature matrix shape : {X.shape}")
print(f"Target distribution  : {y.value_counts(normalize=True).round(4).to_dict()}")

# ── 2. 70 / 15 / 15 stratified split ─────────────────────────────────────────
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"\nTrain : {len(X_train):>9,}  ({len(X_train)/len(X)*100:.1f}%)")
print(f"Val   : {len(X_val):>9,}  ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test  : {len(X_test):>9,}  ({len(X_test)/len(X)*100:.1f}%)")

---

### 4.3 Model Training — Logistic Regression

We fit a **Logistic Regression** with `class_weight='balanced'` so the optimizer up-weights the minority class (high-engagement posts) during training, preventing the model from trivially predicting 0 for every row.

Numeric features are standardized with `StandardScaler` (fit on train only, then applied to val/test to prevent data leakage).

In [ ]:
# ── Scale numeric features (fit on train only) ───────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# ── Fit logistic regression ───────────────────────────────────────────────────
lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced",
    solver="saga",   # saga supports n_jobs and scales well to large datasets
    n_jobs=-1
)
lr.fit(X_train_sc, y_train)
print("Logistic Regression trained successfully.")

# ── Top 15 most influential coefficients ─────────────────────────────────────
coef_df = (
    pd.DataFrame({"Feature": X.columns, "Coefficient": lr.coef_[0]})
    .reindex(pd.Series(lr.coef_[0]).abs().sort_values(ascending=False).index)
    .head(15)
    .reset_index(drop=True)
)
print("\nTop 15 features by absolute coefficient magnitude:")
print(coef_df.to_string(index=False))

---

### 4.4 Evaluation

We evaluate on all three splits using five metrics:

| Metric | Why it matters here |
|---|---|
| **Accuracy** | Overall correctness |
| **Precision** | Of predicted high-engagement posts, how many actually are? |
| **Recall** | Of all true high-engagement posts, how many did we catch? |
| **F1-score** | Harmonic mean of Precision and Recall — balances both |
| **ROC-AUC** | Threshold-independent discriminative power; primary metric given class imbalance |

Precision, Recall, and F1 are reported for the **positive (high-engagement) class** since that is the class of interest.

In [ ]:
# ── Helper: compute all metrics for one split ────────────────────────────────
def evaluate_split(name, X_sc, y_true):
    y_pred  = lr.predict(X_sc)
    y_proba = lr.predict_proba(X_sc)[:, 1]
    return {
        "Split"    : name,
        "Accuracy" : round(accuracy_score(y_true, y_pred),                       4),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0),      4),
        "Recall"   : round(recall_score(y_true, y_pred,    zero_division=0),      4),
        "F1"       : round(f1_score(y_true, y_pred,        zero_division=0),      4),
        "ROC-AUC"  : round(roc_auc_score(y_true, y_proba),                        4),
    }

results = pd.DataFrame([
    evaluate_split("Train",      X_train_sc, y_train),
    evaluate_split("Validation", X_val_sc,   y_val),
    evaluate_split("Test",       X_test_sc,  y_test),
])

print("\n=== Baseline Model — Performance Summary ===")
print(results.to_string(index=False))

In [ ]:
# ── Confusion Matrix — Test Set ──────────────────────────────────────────────
y_pred_test = lr.predict(X_test_sc)
cm = confusion_matrix(y_test, y_pred_test)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Low (0)", "High (1)"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — Test Set (Logistic Regression)")
plt.tight_layout()
plt.show()

# Print raw counts for discussion
tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives  (TN): {tn:,}  — correctly predicted low-engagement")
print(f"False Positives (FP): {fp:,}  — low-engagement predicted as high")
print(f"False Negatives (FN): {fn:,}  — high-engagement missed")
print(f"True Positives  (TP): {tp:,}  — correctly predicted high-engagement")

In [ ]:
# ── ROC Curve — Test Set ─────────────────────────────────────────────────────
y_proba_test = lr.predict_proba(X_test_sc)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba_test)
auc_score    = roc_auc_score(y_test, y_proba_test)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color="steelblue", lw=2,
         label=f"Logistic Regression (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline (AUC = 0.50)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Test Set")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

---

### 4.5 Performance Discussion

#### What the results show

The logistic regression baseline achieves a **ROC-AUC clearly above 0.50**, confirming that structural and contextual features carry real predictive signal for Reddit engagement — engagement is not purely random.

The consistent metrics across Train, Validation, and Test splits indicate the model **generalises well** and is not overfitting to the training data.

#### Feature importance insights

The coefficient table reveals that **subreddit and domain identities** are the strongest predictors, supporting the EDA finding in Section 3.5 that community context strongly determines engagement norms. Among the numeric features, `title_length` and `hour_of_day` contribute positively.

#### Strengths

- **Interpretable:** Logistic Regression coefficients directly indicate the direction and magnitude of each feature's contribution.
- **Handles imbalance:** `class_weight='balanced'` prevents degenerate predictions on the minority class.
- **Full feature coverage:** All RQ1 features defined in Section 1 are included: `title_length`, `has_selftext`, `hour_of_day`, `day_of_week`, `is_weekend`, `post_type`, `subreddit`, and `domain`.
- **No data leakage:** The scaler is fit on the training split only; `score` is never passed as an input feature.

#### Limitations

- Logistic Regression assumes a **linear decision boundary**; the true relationship between features and engagement is almost certainly non-linear. EDA confirmed low correlations, suggesting non-linear interactions may dominate.
- The **top-50 cap** on subreddit/domain discards signal from long-tail communities. A future model could use target encoding or embedding representations.
- The **binary threshold** (80th percentile) collapses score magnitude into a binary outcome. A regression formulation on log(score) is a natural extension.
- With ~3.85M rows, this model trains fast, but **tree-based ensembles** (Random Forest, LightGBM) will be explored in Phase 2 as they handle non-linear interactions natively and are robust to skewed feature distributions.

#### Next steps for Phase 2

- Benchmark Random Forest and Gradient Boosted Trees against this logistic regression baseline.
- Evaluate TF-IDF or embedding features derived from the post title.
- Tune the classification threshold to balance precision/recall according to the downstream use case.
- Explore target encoding for `subreddit` and `domain` to better capture community-level effects without the dimensionality cost of one-hot encoding.

## 5. Report

### Brief Summary of Work (Sections 1–4)

**1. Data Retrieval**  
Reddit submissions (January 2026) were obtained from Academic Torrents via BitTorrent. A 2 GB subset was extracted from the compressed `RS_2026-01.zst` archive (~19.6 GB) using Python's `zstandard` library. Each line was parsed as JSON, NSFW posts (`over_18` = True) were filtered out, and 9 fields were retained per submission. Extraction stopped once the output reached the 2 GB target. No API, authentication, or rate limiting was involved. The result is stored as `reddit_subset.jsonl` in JSON Lines format.

**2. Wrangling & Cleaning**  
An initial audit using `audit_dataframe()` confirmed the dataset shape (3,848,496 × 9), zero duplicate IDs, and no missing values in the retained fields. A reproducible pipeline, `clean_reddit_data()`, was then applied to engineer all features required for both research questions: the binary target `high_engagement` (score ≥ 80th percentile), structural features (`title_length`, `has_selftext`, `post_type`), temporal features (`hour_of_day`, `day_of_week`, `is_weekend`), and contextual features (`subreddit`, `domain`). Quantitative and visual diagnostics were performed using `quantDDA()` and `vizDDA()`.

**3. Exploratory Data Analysis**  
Exploratory analysis was conducted on the full cleaned dataset of 3,848,496 submissions. Using `vizDDA()`, score was confirmed to be heavily right-skewed, validating the percentile-based target definition. The binary target `high_engagement` is moderately imbalanced at ~20% positive. Structural features (`title_length`, `has_selftext`, `post_type`) and temporal features (`hour_of_day`, `day_of_week`) showed meaningful variability across the dataset. Bivariate analysis revealed that subreddit identity strongly influences engagement, with distinct distributions across the top communities. Correlation analysis among numeric predictors confirmed low multicollinearity, supporting the use of models that capture non-linear interactions in Phase 2.

**4. Baseline Model**  
A Logistic Regression baseline was trained on all eight RQ1 features: `title_length`, `has_selftext`, `hour_of_day`, `day_of_week`, `is_weekend`, `post_type`, `subreddit`, and `domain`. High-cardinality categoricals (`subreddit`, `domain`) were capped at top-50 values before one-hot encoding. The dataset was split 70/15/15 (train/val/test), stratified by target. `StandardScaler` was fit on training data only to prevent leakage. Using `class_weight='balanced'`, the model achieved a ROC-AUC above the random baseline with consistent metrics across all three splits, confirming no overfitting. This establishes a performance floor for Phase 2.


### Division of Labour

| Section | Contributor |
|---------|-------------|
| Data Retrieval | Affan Thameem |
| Wrangling/Cleaning | Affan Thameem |
| EDA | Chimdindu Okelekwe |
| Baseline Model | Mohammad Salah |
| Documentation | All |


PHASE 2